# Credit Card Fraud Detection
**Goal:** flag fraudulent card transactions early enough for an operations team to act, while keeping the number of false alerts manageable.

**Data:** 284,807 European card transactions over two days (ULB / Kaggle public dataset). Features `V1–V28` are anonymized PCA components; `Time`, `Amount` and `Class` (1 = fraud) are raw.

**Approach**
1. SQL exploration (DuckDB) to size the problem and find patterns
2. Time-based train/test split (train on earlier transactions, test on later ones — mirrors production)
3. Baseline logistic regression vs. gradient-boosted trees (XGBoost)
4. Choose an alert threshold using a simple cost model, and translate results into business terms

In [1]:
import json, warnings
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             confusion_matrix, precision_score, recall_score)
from xgboost import XGBClassifier
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
ROOT = Path("..")
IMG = ROOT / "images"; IMG.mkdir(exist_ok=True)
con = duckdb.connect()
con.execute("CREATE VIEW tx AS SELECT *, CAST(FLOOR(Time/3600) % 24 AS INTEGER) AS hour_of_day FROM read_csv_auto('../data/creditcard.csv')");

## 1. Size of the problem (SQL)

In [2]:
con.execute("""
SELECT COUNT(*) AS transactions, SUM(Class) AS fraud_cases,
       ROUND(100.0*AVG(Class),3) AS fraud_rate_pct,
       ROUND(SUM(Amount*Class),2) AS fraud_amount
FROM tx""").df()

   transactions  fraud_cases  fraud_rate_pct  fraud_amount
0        284807        492.0           0.173      60127.97

Fraud is **0.17% of transactions** — a classic needle-in-a-haystack problem. Accuracy is meaningless here (predicting 'never fraud' is 99.8% accurate), so we evaluate with **precision, recall and PR-AUC**.

In [3]:
hourly = con.execute("""
SELECT hour_of_day, COUNT(*) AS transactions, SUM(Class) AS fraud_cases,
       100.0*AVG(Class) AS fraud_rate_pct
FROM tx GROUP BY 1 ORDER BY 1""").df()
fig, ax = plt.subplots(figsize=(10,4))
colors = ["#1c5cab" if r > 0.3 else "#86b6ef" for r in hourly.fraud_rate_pct]
ax.bar(hourly.hour_of_day, hourly.fraud_rate_pct, color=colors)
ax.axhline(0.173, ls="--", color="#7a7974", lw=1); ax.text(23, 0.2, "average 0.17%", ha="right", color="#52514e")
ax.set_xlabel("Hour of day"); ax.set_ylabel("Fraud rate (%)")
ax.set_title("Fraud rate by hour of day: overnight hours run up to 10x the average")
fig.tight_layout(); fig.savefig(IMG/"fraud_rate_by_hour.png", dpi=110); plt.show()
hourly.sort_values("fraud_rate_pct", ascending=False).head(5)

   hour_of_day  transactions  fraud_cases  fraud_rate_pct
2            2          3328         57.0        1.712740
4            4          2209         23.0        1.041195
3            3          3492         17.0        0.486827
5            5          2990         11.0        0.367893
7            7          7243         23.0        0.317548

![Fraud rate by hour](https://raw.githubusercontent.com/shanmukha-bhimireddy/credit-card-fraud-detection/main/images/fraud_rate_by_hour.png)

In [4]:
bands = con.execute("""
SELECT CASE WHEN Amount=0 THEN '0. $0' WHEN Amount<10 THEN '1. <$10' WHEN Amount<100 THEN '2. $10-99'
            WHEN Amount<500 THEN '3. $100-499' ELSE '4. $500+' END AS amount_band,
       COUNT(*) AS transactions, SUM(Class) AS fraud_cases,
       ROUND(100.0*AVG(Class),3) AS fraud_rate_pct, ROUND(SUM(Amount*Class),2) AS fraud_amount
FROM tx GROUP BY 1 ORDER BY 1""").df()
bands

   amount_band  transactions  fraud_cases  fraud_rate_pct  fraud_amount
0        0. $0          1825         27.0           1.479          0.00
1      1. <$10         95489        222.0           0.232        458.90
2    2. $10-99        130108        113.0           0.087       6560.36
3  3. $100-499         47893         95.0           0.198      21913.81
4     4. $500+          9492         35.0           0.369      31194.90

## 2. Time-based split
The first 80% of transactions (by time) are used for training and the last 20% for testing, so the model is always evaluated on *future* data.

In [5]:
df = con.execute("SELECT * EXCLUDE (hour_of_day) FROM tx ORDER BY Time").df()
df["log_amount"] = np.log1p(df.Amount)
df["hour_of_day"] = (df.Time // 3600) % 24
features = [f"V{i}" for i in range(1,29)] + ["log_amount","hour_of_day"]
cut = int(len(df)*0.8)
train, test = df.iloc[:cut], df.iloc[cut:]
X_tr, y_tr, X_te, y_te = train[features], train.Class, test[features], test.Class
print(f"Train: {len(train):,} tx, {y_tr.sum()} fraud | Test: {len(test):,} tx, {y_te.sum()} fraud")

Train: 227,845 tx, 417 fraud | Test: 56,962 tx, 75 fraud


## 3. Models

In [6]:
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
logit.fit(X_tr, y_tr)
p_logit = logit.predict_proba(X_te)[:,1]

spw = (y_tr==0).sum()/(y_tr==1).sum()
xgb = XGBClassifier(tree_method="hist", n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="aucpr",
                    random_state=42, n_jobs=4)
xgb.fit(X_tr, y_tr)
p_xgb = xgb.predict_proba(X_te)[:,1]

summary = pd.DataFrame({
  "model": ["Logistic regression (baseline)", "XGBoost"],
  "PR-AUC": [average_precision_score(y_te,p_logit), average_precision_score(y_te,p_xgb)],
  "ROC-AUC": [roc_auc_score(y_te,p_logit), roc_auc_score(y_te,p_xgb)],
}).round(3)
summary

                            model  PR-AUC  ROC-AUC
0  Logistic regression (baseline)   0.759    0.987
1                         XGBoost   0.794    0.986

In [7]:
fig, ax = plt.subplots(figsize=(6,5))
for name, p in [("Logistic regression", p_logit), ("XGBoost", p_xgb)]:
    pr, rc, _ = precision_recall_curve(y_te, p)
    ax.plot(rc, pr, label=f"{name} (PR-AUC {average_precision_score(y_te,p):.2f})")
ax.set_xlabel("Recall (share of fraud caught)"); ax.set_ylabel("Precision (share of alerts that are fraud)")
ax.set_title("Precision-recall: XGBoost vs. baseline"); ax.legend()
fig.tight_layout(); plt.show()

## 4. Choosing an alert threshold (business view)
Assumptions for a simple cost model:
- A **missed fraud** costs the full transaction amount (chargeback)
- Each **alert** costs **$5** of analyst review time

We pick the threshold that minimizes total cost on the test period.

In [8]:
REVIEW_COST = 5.0
amt = test.Amount.values; y = y_te.values
rows = []
for t in np.round(np.arange(0.05, 1.0, 0.05), 2):
    alert = p_xgb >= t
    missed = amt[(~alert) & (y==1)].sum()
    rows.append({"threshold": t, "alerts": int(alert.sum()),
                 "fraud_caught": int((alert & (y==1)).sum()),
                 "precision": precision_score(y, alert, zero_division=0),
                 "recall": recall_score(y, alert),
                 "missed_fraud_$": missed,
                 "total_cost_$": missed + REVIEW_COST*alert.sum()})
th = pd.DataFrame(rows)
best = th.loc[th["total_cost_$"].idxmin()]
no_model_cost = amt[y==1].sum()
th.round(3).iloc[::2]

    threshold  alerts  fraud_caught  precision  recall  missed_fraud_$  total_cost_$
0        0.05     488            64      0.131   0.853         1736.12       4176.12
2        0.15     190            61      0.321   0.813         2385.48       3335.48
4        0.25     125            60      0.480   0.800         2397.79       3022.79
6        0.35     104            58      0.558   0.773         2636.05       3156.05
8        0.45      89            58      0.652   0.773         2636.05       3081.05
10       0.55      78            57      0.731   0.760         2638.27       3028.27
12       0.65      75            57      0.760   0.760         2638.27       3013.27
14       0.75      69            57      0.826   0.760         2638.27       2983.27
16       0.85      66            57      0.864   0.760         2638.27       2968.27
18       0.95      62            55      0.887   0.733         3882.40       4192.40

In [9]:
t = best.threshold
alert = p_xgb >= t
cm = confusion_matrix(y, alert)
fig, ax = plt.subplots(figsize=(4.5,4))
sns.heatmap(cm, annot=True, fmt=",", cmap="Blues", cbar=False,
            xticklabels=["No alert","Alert"], yticklabels=["Legit","Fraud"], ax=ax)
ax.set_title(f"XGBoost at threshold {t:.2f}"); ax.set_xlabel("Model"); ax.set_ylabel("Actual")
fig.tight_layout(); plt.show()

fraud_amt_caught = amt[alert & (y==1)].sum()
results = {
  "test_transactions": int(len(y)), "test_fraud_cases": int(y.sum()),
  "pr_auc_logit": round(float(average_precision_score(y,p_logit)),3),
  "pr_auc_xgb": round(float(average_precision_score(y,p_xgb)),3),
  "threshold": float(t), "alerts": int(alert.sum()),
  "precision": round(float(best.precision),3), "recall": round(float(best.recall),3),
  "fraud_amount_total": round(float(amt[y==1].sum()),2),
  "fraud_amount_caught": round(float(fraud_amt_caught),2),
  "cost_no_model": round(float(no_model_cost),2),
  "cost_with_model": round(float(best["total_cost_$"]),2),
}
(ROOT/"results").mkdir(exist_ok=True)
json.dump(results, open(ROOT/"results"/"metrics.json","w"), indent=2)
results

{'test_transactions': 56962,
 'test_fraud_cases': 75,
 'pr_auc_logit': 0.759,
 'pr_auc_xgb': 0.794,
 'threshold': 0.9,
 'alerts': 65,
 'precision': 0.877,
 'recall': 0.76,
 'fraud_amount_total': 7729.26,
 'fraud_amount_caught': 5090.99,
 'cost_no_model': 7729.26,
 'cost_with_model': 2963.27}

In [10]:
imp = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(6,4))
imp[::-1].plot.barh(ax=ax, color="#2c7fb8"); ax.set_title("Top 10 features (XGBoost gain)")
fig.tight_layout(); plt.show()

## 5. Takeaways
- Fraud is rare (0.17%) and concentrated in **overnight hours**, where the fraud rate reaches up to 10× the daily average — a case for tighter overnight rules.
- **XGBoost beats the logistic-regression baseline** on PR-AUC (0.79 vs. 0.76), the right metric for rare events.
- At the cost-optimal threshold, **88% of alerts are real fraud**, 76% of fraud cases are caught, and total cost (missed fraud + reviews) is **62% lower** than with no model — see `results/metrics.json`.
- **Next steps:** add customer-level velocity features (transactions per hour, new merchant), calibrate scores, and monitor drift weekly.